# SAE Dual-Pipeline Visualization

这个 notebook 读取 `scripts/run_sae_pipeline.py` 生成的 `pipeline_summary.json`，对 `standard` 和 `attnres` 两个模型的 SAE 训练曲线与结构评估结果做并排可视化。

## 1. 定位结果目录

这一步不是可视化本身，而是先自动定位仓库根目录和 `pipeline_summary.json`。

你可以把这里理解成“告诉 notebook 去哪里找刚刚跑完的 SAE 双模型结果”。如果路径不对，后面的图都不会画出来。

In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

try:
    import pandas as pd
except Exception:
    pd = None

plt.rcParams["axes.unicode_minus"] = False
MODEL_COLORS = {
    "standard": "#1f77b4",
    "attnres": "#ff7f0e",
}


def find_repo_root(start: Path | None = None) -> Path:
    candidate = Path.cwd() if start is None else start
    for path in [candidate, *candidate.parents]:
        if (path / "stream_analysis").exists() and (path / "scripts").exists():
            return path
    raise FileNotFoundError("Could not locate the repo root from the current working directory.")


REPO_ROOT = find_repo_root()
SUMMARY_PATH = REPO_ROOT / "outputs" / "sae_dual_pipeline" / "pipeline_summary.json"
REPO_ROOT, SUMMARY_PATH

## 2. 加载总结果清单

这一步会读取 `pipeline_summary.json`，然后把两个模型最关键的输出路径和最优验证集重构误差整理成一个表。

这张表的作用是先做一次“结果核对”：确认 `standard` 和 `attnres` 的 checkpoint、SAE 输出目录、评估 summary 都已经生成。

In [ ]:
if not SUMMARY_PATH.is_file():
    raise FileNotFoundError(
        f"Missing pipeline summary: {SUMMARY_PATH}\n"
        "先运行 accelerate launch scripts/run_sae_pipeline.py ... 生成双模型 SAE 结果。"
    )

with SUMMARY_PATH.open("r", encoding="utf-8") as handle:
    pipeline_summary = json.load(handle)

overview_rows = []
for model_type, payload in pipeline_summary["models"].items():
    overview_rows.append(
        {
            "model_type": model_type,
            "checkpoint": payload["checkpoint_path"],
            "best_val_recon_mse": payload["best_val_recon_mse"],
            "sae_dir": payload["sae_dir"],
            "eval_summary_path": payload.get("eval_summary_path", ""),
        }
    )

if pd is not None:
    display(pd.DataFrame(overview_rows))
else:
    overview_rows

## 3. 读取训练曲线和评估 summary

这一步还是准备数据，不会直接画图。它会把每个模型的 `metrics.csv` 和 `eval_summary.json` 读进内存。

后面的所有可视化都依赖这一步输出的 `train_curves` 和 `eval_payloads`。

In [ ]:
def read_metrics_csv(path: str | Path):
    rows = []
    with Path(path).open("r", encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            rows.append({key: float(value) for key, value in row.items()})
    return rows


def load_eval_summary(path: str | Path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


train_curves = {
    model_type: read_metrics_csv(payload["metrics_path"])
    for model_type, payload in pipeline_summary["models"].items()
}
eval_payloads = {
    model_type: load_eval_summary(payload["eval_summary_path"])
    for model_type, payload in pipeline_summary["models"].items()
}

list(train_curves.keys())

## 4. 训练曲线可视化

这张图展示两个模型在 SAE 训练过程中的重构误差变化：

- 左图是 `train_recon_mse`，看训练集上是否稳定下降
- 右图是 `val_recon_mse`，看 held-out 验证激活上的泛化情况

如果右图持续下降且没有明显发散，通常说明这组 SAE 超参数是可用的；如果训练误差很低但验证误差不降，说明可能过拟合或训练数据分布不够稳。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=False)
for axis, metric_key, title in [
    (axes[0], "train_recon_mse", "Train Reconstruction MSE"),
    (axes[1], "val_recon_mse", "Validation Reconstruction MSE"),
]:
    for model_type, rows in train_curves.items():
        steps = [int(row["step"]) for row in rows]
        values = [row[metric_key] for row in rows]
        axis.plot(steps, values, marker="o", label=model_type, color=MODEL_COLORS.get(model_type, "#4c4c4c"))
    axis.set_title(title)
    axis.set_xlabel("step")
    axis.set_ylabel(metric_key)
    axis.grid(alpha=0.3)
    axis.legend()
plt.tight_layout()
plt.show()

## 5. SAE 结构评估指标对比

这一部分会把两个模型最终 SAE 的核心结构指标拆成多张独立柱状图分别比较：

- `recon_mse`：原始重构误差，越低越好
- `normalized_recon_mse`：按输入方差归一化后的重构误差，越低越好
- `avg_l0`：平均激活 latent 数，反映稀疏程度
- `dead_latent_frac`：死特征比例，越低通常越好

这样做的好处是不会把量纲不同的指标挤在同一张图里，更容易单独判断某一个指标上到底是哪边更好。

In [ ]:
metric_specs = [
    ("recon_mse", "Reconstruction MSE", "Lower is better; raw reconstruction error."),
    ("normalized_recon_mse", "Normalized Reconstruction MSE", "Lower is better; normalized by input variance."),
    ("avg_l0", "Average L0", "Average number of active latents per sample."),
    ("dead_latent_frac", "Dead Latent Fraction", "Lower is better; fraction of inactive latents."),
]
model_order = list(pipeline_summary["models"].keys())

for metric_key, title, subtitle in metric_specs:
    values = [float(eval_payloads[model_type]["summary"][metric_key]) for model_type in model_order]
    x = np.arange(len(model_order))
    colors = [MODEL_COLORS.get(model_type, "#4c4c4c") for model_type in model_order]

    fig, ax = plt.subplots(figsize=(7, 4.8))
    bars = ax.bar(x, values, width=0.6, color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(model_order)
    ax.set_ylabel(metric_key)
    ax.set_title(f"{title}\n{subtitle}")
    ax.grid(axis="y", alpha=0.3)

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.4f}",
            ha="center",
            va="bottom",
        )

    plt.tight_layout()
    plt.show()

## 6. Decoder Overlap 热图

这张热图展示 SAE decoder 不同 latent 原子之间的余弦相似度。

你可以把它理解成“不同特征之间是不是在重复表示类似方向”。

- 对角线附近或大面积高亮：说明很多 latent 很相似，特征有冗余
- 整体更分散、非对角线不那么亮：说明 decoder 原子更分化

这张图主要是在看特征字典是否“分得开”。

注意：如果 latent 数太大，评估阶段可能不会把完整 `heatmap` 写进 `eval_summary.json`。这时候下面会自动退化成 `top_overlapping_pairs` 条形图，而不是只显示一个空标题。

In [ ]:
fig, axes = plt.subplots(1, len(model_order), figsize=(7 * len(model_order), 5))
if len(model_order) == 1:
    axes = [axes]

for axis, model_type in zip(axes, model_order):
    decoder_overlap_payload = eval_payloads[model_type]["decoder_overlap"]
    decoder_overlap = decoder_overlap_payload.get("heatmap")
    if decoder_overlap is not None:
        image = axis.imshow(np.asarray(decoder_overlap, dtype=float), aspect="auto", interpolation="nearest")
        axis.set_title(f"{model_type}: decoder overlap heatmap")
        axis.set_xlabel("latent j")
        axis.set_ylabel("latent i")
        fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
        continue

    top_pairs = decoder_overlap_payload.get("top_overlapping_pairs") or []
    if not top_pairs:
        axis.text(0.5, 0.5, "decoder overlap data unavailable", ha="center", va="center")
        axis.set_title(f"{model_type}: decoder overlap unavailable")
        axis.set_xticks([])
        axis.set_yticks([])
        continue

    labels = [f"{int(row['latent_i'])}-{int(row['latent_j'])}" for row in top_pairs]
    values = [float(row["score"]) for row in top_pairs]
    y = np.arange(len(values))
    axis.barh(y, values, color=MODEL_COLORS.get(model_type, "#4c4c4c"))
    axis.set_yticks(y)
    axis.set_yticklabels(labels)
    axis.invert_yaxis()
    axis.set_xlabel("cosine overlap")
    axis.set_ylabel("latent pair")
    axis.set_title(f"{model_type}: top decoder overlap pairs")
    axis.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Coactivation 热图

这张热图展示 latent 两两之间在样本上共同激活的频率。

它回答的是：哪些特征经常一起出现，是否形成了稳定的组合结构。

- 某些局部块特别亮：说明有一组 latent 经常成团共现
- 整体比较稀疏：说明特征之间更独立

这张图更偏向看“特征之间的协同关系”，而不是单个特征本身质量。

In [ ]:
fig, axes = plt.subplots(1, len(model_order), figsize=(7 * len(model_order), 5))
if len(model_order) == 1:
    axes = [axes]

for axis, model_type in zip(axes, model_order):
    coactivation = eval_payloads[model_type]["coactivation"].get("heatmap")
    if coactivation is None:
        axis.axis("off")
        axis.set_title(f"{model_type}: coactivation heatmap unavailable")
        continue
    image = axis.imshow(np.asarray(coactivation, dtype=float), aspect="auto", interpolation="nearest")
    axis.set_title(f"{model_type}: coactivation")
    axis.set_xlabel("latent j")
    axis.set_ylabel("latent i")
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()